<a href="https://colab.research.google.com/github/devangc006/Ai-ML/blob/main/LGS_RGR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub flask scikit-learn pandas

import glob, os, socket, threading
import pandas as pd
from flask import Flask, Response, jsonify, request
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- 1. LOAD DATA (Kaggle, with sklearn fallback) ----------------
def load_data():
    try:
        import kagglehub
        path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
        df = pd.read_csv(glob.glob(os.path.join(path, "*.csv"))[0])
        df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])
        y = (df["diagnosis"] == "M").astype(int)      # 1 = Malignant
        X = df.drop(columns="diagnosis")
        print("Loaded from Kaggle:", df.shape)
        return X, y
    except Exception as e:
        print("Kaggle failed (%s). Using sklearn's copy of the same dataset." % e)
        from sklearn.datasets import load_breast_cancer
        d = load_breast_cancer(as_frame=True)
        X = d.data.copy()
        def rename(col):
            if col.startswith("mean "):
                base, suf = col[5:], "_mean"
            elif col.startswith("worst "):
                base, suf = col[6:], "_worst"
            else:
                base, suf = col.replace(" error", ""), "_se"
            return base.replace("fractal dimension", "fractal_dimension") + suf
        X.columns = [rename(c) for c in X.columns]
        return X, 1 - d.target                        # flip so 1 = Malignant

# ---------------- 2. TRAIN + EVALUATE ----------------
X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(solver="liblinear", max_iter=5000))])
grid = GridSearchCV(
    pipe,
    {"clf__C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100],
     "clf__penalty": ["l1", "l2"],
     "clf__class_weight": [None, "balanced"]},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
model = grid.best_estimator_
print("Best params:", grid.best_params_)
print("CV accuracy: %.4f" % grid.best_score_)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred),
           "roc_auc": roc_auc_score(y_test, proba)}
print("Test metrics:", {k: round(float(v), 4) for k, v in metrics.items()})
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["Benign", "Malignant"]))

def first_correct(label):
    idx = X_test.index[(y_test == label) & (pred == label)][0]
    return {k: float(v) for k, v in X_test.loc[idx].items()}

FEATURES = list(X.columns)
META = {"features": FEATURES,
        "samples": {"malignant": first_correct(1), "benign": first_correct(0)},
        "metrics": {k: round(float(v), 4) for k, v in metrics.items()}}

# ---------------- 3. HTML UI ----------------
HTML = """<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Breast Cancer Classifier</title>
<style>
  body { font-family: system-ui, sans-serif; margin: 0; background: #f4f6f8; color: #222; }
  header { background: #1f3a5f; color: #fff; padding: 16px 24px; }
  main { display: flex; flex-wrap: wrap; gap: 20px; padding: 20px; }
  #form { flex: 2 1 560px; background: #fff; padding: 16px; border-radius: 10px; }
  #result { flex: 1 1 260px; background: #fff; padding: 16px; border-radius: 10px;
            position: sticky; top: 10px; align-self: flex-start; }
  fieldset { border: 1px solid #ddd; border-radius: 8px; margin-bottom: 14px; }
  legend { font-weight: 600; padding: 0 6px; }
  .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: 10px; }
  label { font-size: 12px; display: flex; flex-direction: column; gap: 3px; }
  input { padding: 6px; border: 1px solid #bbb; border-radius: 6px; }
  button { padding: 8px 14px; border: 0; border-radius: 6px; background: #1f3a5f;
           color: #fff; cursor: pointer; margin-right: 6px; }
  button.alt { background: #6b7c93; }
  #label { font-size: 28px; font-weight: 700; margin: 10px 0; }
  .bar { height: 18px; background: #e3e8ee; border-radius: 9px; overflow: hidden; }
  #fill { height: 100%; width: 0; background: #2e9e5b; transition: width .25s, background .25s; }
  .benign { color: #2e9e5b; } .malignant { color: #c0392b; }
  small { color: #666; }
</style></head>
<body>
<header><h1 style="margin:0;font-size:20px">Breast Cancer Classifier (Logistic Regression)</h1></header>
<main>
  <section id="form">
    <div style="margin-bottom:12px">
      <button onclick="loadSample('benign')">Load benign sample</button>
      <button onclick="loadSample('malignant')">Load malignant sample</button>
      <button class="alt" onclick="clearAll()">Clear</button>
    </div>
    <div id="groups"></div>
  </section>
  <aside id="result">
    <h3 style="margin-top:0">Prediction</h3>
    <div id="label">Waiting for input...</div>
    <div class="bar"><div id="fill"></div></div>
    <p id="probs"></p><hr>
    <small id="metrics"></small>
    <p><small>Educational demo only, not a medical diagnosis.</small></p>
  </aside>
</main>
<script>
let META = null; const inputs = {}; let timer = null;
async function init() {
  META = await (await fetch("meta")).json();
  const groups = { mean: [], se: [], worst: [] };
  META.features.forEach(f => {
    const g = f.endsWith("_mean") ? "mean" : f.endsWith("_se") ? "se" : "worst";
    groups[g].push(f);
  });
  const titles = { mean: "Mean values", se: "Standard error", worst: "Worst values" };
  const root = document.getElementById("groups");
  for (const g of ["mean", "se", "worst"]) {
    const fs = document.createElement("fieldset");
    fs.innerHTML = "<legend>" + titles[g] + "</legend><div class='grid'></div>";
    groups[g].forEach(f => {
      const lab = document.createElement("label");
      lab.textContent = f.replace(/_/g, " ");
      const inp = document.createElement("input");
      inp.type = "number"; inp.step = "any";
      inp.addEventListener("input", schedule);
      inputs[f] = inp; lab.appendChild(inp);
      fs.querySelector(".grid").appendChild(lab);
    });
    root.appendChild(fs);
  }
  const m = META.metrics;
  document.getElementById("metrics").textContent =
    "Test set: accuracy " + m.accuracy + ", precision " + m.precision +
    ", recall " + m.recall + ", AUC " + m.roc_auc;
}
function loadSample(kind) {
  const s = META.samples[kind];
  for (const f in s) inputs[f].value = s[f];
  predict();
}
function clearAll() {
  Object.values(inputs).forEach(i => i.value = "");
  const l = document.getElementById("label");
  l.textContent = "Waiting for input..."; l.className = "";
  document.getElementById("fill").style.width = "0";
  document.getElementById("probs").textContent = "";
}
function schedule() { clearTimeout(timer); timer = setTimeout(predict, 250); }
async function predict() {
  const payload = {};
  for (const f of META.features) {
    if (inputs[f].value === "") return;
    payload[f] = inputs[f].value;
  }
  const res = await fetch("predict", { method: "POST",
    headers: { "Content-Type": "application/json" }, body: JSON.stringify(payload) });
  const out = await res.json();
  const label = document.getElementById("label");
  if (out.error) { label.textContent = out.error; return; }
  label.textContent = out.label;
  label.className = out.label === "Malignant" ? "malignant" : "benign";
  const pct = out.probability_malignant * 100;
  const fill = document.getElementById("fill");
  fill.style.width = pct + "%";
  fill.style.background = pct >= 50 ? "#c0392b" : "#2e9e5b";
  document.getElementById("probs").innerHTML =
    "Malignant: <b>" + pct.toFixed(2) + "%</b><br>Benign: <b>" +
    (out.probability_benign * 100).toFixed(2) + "%</b>";
}
init();
</script></body></html>"""

# ---------------- 4. FLASK SERVER ----------------
app = Flask(__name__)

@app.route("/")
def home():
    return Response(HTML, mimetype="text/html")

@app.route("/meta")
def get_meta():
    return jsonify(META)

@app.route("/predict", methods=["POST"])
def predict_route():
    data = request.get_json(silent=True) or {}
    try:
        row = [float(data[f]) for f in FEATURES]
    except (KeyError, TypeError, ValueError):
        return jsonify(error="All 30 features must be numeric."), 400
    p = float(model.predict_proba(pd.DataFrame([row], columns=FEATURES))[0, 1])
    return jsonify(label="Malignant" if p >= 0.5 else "Benign",
                   probability_malignant=round(p, 4),
                   probability_benign=round(1 - p, 4))

# ---------------- 5. LAUNCH IN COLAB ----------------
def free_port():
    s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
    return port

from google.colab import output
PORT = free_port()   # fresh port each run, so re-running the cell never clashes
threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT, use_reloader=False),
                 daemon=True).start()
output.serve_kernel_port_as_iframe(PORT, height=900)

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Loaded from Kaggle: (569, 31)
Best params: {'clf__C': 0.05, 'clf__class_weight': None, 'clf__penalty': 'l2'}
CV accuracy: 0.9780
Test metrics: {'accuracy': 0.9825, 'precision': 1.0, 'recall': 0.9524, 'f1': 0.9756, 'roc_auc': 0.998}
Confusion matrix:
 [[72  0]
 [ 2 40]]
              precision    recall  f1-score   support

      Benign       0.97      1.00      0.99        72
   Malignant       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



<IPython.core.display.Javascript object>